## 1. 离线处理部分

### 1.1 clip模型特征抽取（将图像或视频关键帧转化为向量）

#### 关键帧的抽取

In [ ]:
# 视频信息的读取

import cv2
import numpy as np
##提取视频关键帧
def extract_keyframes(video_path, topK=5):
    cap = cv2.VideoCapture(video_path)
    keyframes = []
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    results=[]
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        #每一帧的灰度图
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # 计算帧间差异（直方图或像素差值）
        diff = cv2.absdiff(prev_gray, gray)
        # 镜头切换、大幅动作时 diff_mean 会显著升高,定义为关键正
        diff_mean = np.mean(diff)
        results.append([frame,diff_mean])
        prev_gray = gray
    cap.release()
    results=sorted(results,key=lambda s:s[1],reverse=True)[0:topK]
    return [s[0] for s in results]
if __name__ == "__main__":
    path = "./data/video/3fa8c42c64b7357e4148ae5ac5a7fcf7.mp4"
    frames=extract_keyframes(path)
    print (frames)

#### 关键帧转化为向量

In [ ]:
# huggingface

from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel
import torch

#提取图像向量特征
def normal(vector):
    ss=float(sum([s**2 for s in vector])**0.5)
    return [float(s)/ss for s in vector]

#model_name="Clip"
model_name="D:/models/clip/"
model = CLIPModel.from_pretrained(model_name).to("cuda:0")
processor = CLIPProcessor.from_pretrained(model_name)
#
def extract_img_vector(image):
    #图像部分
    img_model=model.vision_model
    inputs = processor(images=image, return_tensors="pt", padding=True).to("cuda:0")
    img_vector=img_model(**inputs).pooler_output[0]
    img_vector=model.visual_projection(img_vector)
    img_vector= normal(img_vector)
    return img_vector

def extract_text_vector(text):
    #文字部分
    inputs = processor(text=[text],  return_tensors="pt", padding=True).to("cuda:0")
    text_model=model.text_model
    text_vector= text_model(**inputs).pooler_output[0]
    text_vector=model.text_projection(text_vector)
    text_vector= normal(text_vector)
    return text_vector

if __name__ == "__main__":
    data = "./data/image/微信图片_20260629093158_919_10"
    #提取文字对应的向量
    vector_text=extract_text_vector("叶片拼成的水豚噜噜")


    #提取图像对应的向量
    image = Image.open(path)
    vector_img=extract_img_vector(image)

    similar=sum([ s1*s2 for s1,s2 in  zip(vector_img,vector_text)])
    print ("similar",similar)

In [ ]:
# 魔搭镜像

from PIL import Image
import torch
from modelscope import pipeline, Tasks

# ---------------- 改这里为你本地模型文件夹路径 ----------------
LOCAL_MODEL_PATH = r"D:/models/clip/"
# 加载阿里中文CLIP流水线
embedder = pipeline(
    task=Tasks.multi_modal_embedding,
    model=LOCAL_MODEL_PATH,
    local_files_only=True,  # 强制只用本地文件，不联网
    device="cuda:0"
)

def normalize(tensor_vec):
    return tensor_vec / torch.norm(tensor_vec, p=2, keepdim=True)

@torch.no_grad()
def extract_img_vector(image: Image.Image):
    res = embedder({"img": image})
    emb = torch.tensor(res["img_embedding"])
    return normalize(emb).squeeze(0)

@torch.no_grad()
def extract_text_vector(text: str):
    res = embedder({"text": text})
    emb = torch.tensor(res["text_embedding"])
    return normalize(emb).squeeze(0)

if __name__ == "__main__":
    img_path = r"./data/image/微信图片_20260629093158_919_10.jpg"
    text_query = "叶片拼成的水豚噜噜"

    img = Image.open(img_path).convert("RGB")
    vec_img = extract_img_vector(img)
    vec_txt = extract_text_vector(text_query)

    sim = torch.dot(vec_img, vec_txt).item()
    print(f"图文余弦相似度: {sim:.4f}")

### 1.2 将模型生成的视频描述内容转化为向量

In [ ]:
# Requires transformers>=4.51.0
# Requires sentence-transformers>=2.7.0

def normal(vector):
    ss=float(sum([s**2 for s in vector])**0.5)
    return [float(s)/ss for s in vector]

from sentence_transformers import SentenceTransformer

# Load the model

model = SentenceTransformer("F:\\qwen3_embedding")

def extract_qwen_embedding(query):
    query_embeddings = model.encode([query], prompt_name="query")[0]
    return normal(query_embeddings)

### 1.3 提取音频文字

In [ ]:
# ============================== 模块导入 ==============================
from modelscope.pipelines import pipeline   # 导入 ModelScope 的管道工具
from modelscope.utils.constant import Tasks # 导入预定义任务常量（如语音识别）
from convert_mp42wav import *               # 导入自定义模块：将 MP4 转换为 WAV（包含 extract_audio_from_mp4）
from pydub import AudioSegment              # 用于音频切割和处理
import os                                   # 文件和路径操作（原代码缺失，补充以确保 os.path 可用）

# ============================== 全局变量 ==============================
wav_path = "tmp.wav"   # 临时 WAV 文件路径（用于存放从 MP4 提取的音频）

# 初始化语音识别管道（使用 Whisper 模型，路径为本地 F:\Whisper）
inference_pipeline = pipeline(
    task=Tasks.auto_speech_recognition,   # 自动语音识别任务
    model='F:\\Whisper'                  # 指定模型目录
)

# ============================== 音频切割与识别函数 ==============================
def split_wav(input_file: str, segment_length: int = 15, overlap: int = 0):
    """
    将长音频 WAV 文件按固定长度切分为多个片段，并对每个片段进行语音识别，
    最后将所有片段的识别结果拼接返回。

    参数:
        input_file: 输入的 WAV 文件路径
        segment_length: 每个片段的时长（秒），默认 15 秒
        overlap: 片段之间的重叠时长（秒），默认 0（无重叠）

    返回:
        拼接后的完整识别文本
    """
    # 加载 WAV 音频文件
    audio = AudioSegment.from_wav(input_file)

    # 获取基本文件名（不带扩展名，当前未使用，保留原逻辑）
    base_name = os.path.splitext(os.path.basename(input_file))[0]

    # 将时长参数转换为毫秒（pydub 使用毫秒为单位）
    segment_length_ms = segment_length * 1000
    overlap_ms = overlap * 1000
    step_ms = segment_length_ms - overlap_ms   # 每次前进的步长（毫秒）

    # 获取音频总时长（毫秒）
    duration_ms = len(audio)
    result = ""   # 用于累计所有片段的识别文字

    # 按步长循环切割音频
    segment_count = 0
    for start_ms in range(0, duration_ms, step_ms):
        end_ms = start_ms + segment_length_ms   # 当前片段结束时间
        # 若结束时间超出总时长，则截断至末尾
        if end_ms > duration_ms:
            end_ms = duration_ms

        # 提取当前片段（从 start_ms 到 end_ms）
        segment = audio[start_ms:end_ms]
        # 将片段导出为临时文件 tmp2.wav（供识别管道使用）
        segment.export("tmp2.wav", format="wav")

        # 调用语音识别管道，识别该片段内容
        rec_result = inference_pipeline(input="tmp2.wav", language=None)
        # 提取识别结果文本（假设返回列表，第一个元素包含 'text' 字段）
        result += rec_result[0]["text"]

        segment_count += 1   # 计数（未实际使用，保留）

    return result

# ============================== 对外接口函数 ==============================
def extract_audio_text(mp4_path):
    """
    从 MP4 文件中提取音频并转为文字。
    步骤：1. 从 MP4 提取音频为 WAV；2. 将 WAV 切分识别；3. 返回完整文字。
    """
    # 调用 convert_mp42wav 模块中的函数，将 MP4 音频提取到全局 wav_path
    extract_audio_from_mp4(mp4_path, wav_path)
    # 对提取的 WAV 进行切割和识别（片段长度 15 秒，无重叠）
    result = split_wav(wav_path, segment_length=15, overlap=0)
    return result

# ============================== （已注释的测试代码） ==============================
# rec_result = inference_pipeline(input=wav_path, language=None)
# print(rec_result)

### 1.4 视频描述生成

In [ ]:
# ============================== 多模态描述/问答模块（Qwen2.5-VL） ==============================
# 该模块使用 Qwen2.5-VL 多模态大模型，支持图像或视频输入，
# 可生成描述或根据用户问题（query）进行视觉问答。
# 模型从本地路径 F:\qwen2.5-VL 加载。

# ============================== 导入依赖 ==============================
from modelscope import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================== 模型加载（全局初始化） ==============================
# （原代码中定义了一些测试路径，保留但注释说明）
# path = "F:\\video\\1.mp4"        # 示例视频路径（未使用）
# path = "F:\\video\\1.jpg"        # 示例图片路径（未使用）

# 加载处理器（负责图像/视频预处理和文本 tokenization）
processor = AutoProcessor.from_pretrained("F:\\qwen2.5-VL")
# 加载模型（自动选择设备，数据类型自动）
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "F:\\qwen2.5-VL",
    torch_dtype="auto",
    device_map="auto"
)

# ============================== 核心函数：提取文本特征（视觉问答/描述） ==============================
def extract_text_feature(path, query=None):
    """
    根据输入的文件路径（图像或视频）和可选的查询（query），
    调用 Qwen2.5-VL 模型生成对应的文本描述或回答。

    参数:
        path: 图像或视频文件的路径（支持 .mp4 视频和常见图片格式）
        query: 用户问题或提示文本。若为 None，则自动生成通用描述请求。

    返回:
        模型生成的文本字符串。
    """
    # 判断文件类型（视频为 mp4，否则视为图片）
    type_ = "video" if path.endswith("mp4") else "image"

    # 若未提供 query，则根据文件类型生成默认描述问题
    if query is None:
        describe = "描述下这个" + ("视频" if path.endswith("mp4") else "图片")
    else:
        describe = query

    # 构建符合 Qwen2.5-VL 要求的消息格式
    # 注意：content 列表中第一个元素是多媒体输入，第二个是文本提问
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": type_,                         # 媒体类型：video 或 image
                    type_: path,                           # 键为媒体类型，值为文件路径（原代码写法，保留）
                    "max_pixels": 360 * 420,              # 最大像素限制
                    "fps": 1.0,                           # 视频帧率（仅视频有效）
                    # "nframes": 10                       # 可选：指定帧数（被注释）
                },
                {"type": "text", "text": describe},       # 文本提问
            ],
        }
    ]

    # 应用聊天模板，将消息转为模型输入格式的字符串（不 tokenize）
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # 从消息中提取视觉信息（图像/视频）并获取视频处理参数
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages, return_video_kwargs=True
    )

    # 将文本和视觉信息一起送入处理器，得到模型输入张量
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs,
    )
    # 将输入数据移至 GPU（cuda）
    inputs = inputs.to("cuda")

    # 模型推理，生成回答（最大新 token 数 2048）
    generated_ids = model.generate(**inputs, max_new_tokens=2048)

    # 去除输入部分，只保留新生成的 token
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    # 解码生成的 token 为文本
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    # 返回第一个（也是唯一一个）结果
    return output_text[0]

### 1.5 离线特征入库阶段

In [ ]:
# ============================== 模块导入 ==============================
import extract_frame          # 自定义模块：提取视频关键帧
from PIL import Image         # 处理图像
import faiss                  # Facebook 的相似性搜索库，用于向量索引
import numpy as np            # 数值计算
from whoosh.index import create_in   # Whoosh 索引创建
from whoosh.fields import Schema, TEXT, ID   # Whoosh 字段定义
import jieba                  # 中文分词
from whoosh.analysis import Tokenizer, Token  # 自定义分词器基类
import os                     # 文件和路径操作
from tqdm import tqdm         # 进度条
import json                   # JSON 序列化

# ============================== 自定义分词器（用于 Whoosh） ==============================
class JiebaTokenizer(Tokenizer):
    """
    使用结巴分词作为 Whoosh 的分词器。
    将文本按中文词语切分，生成 Token 流。
    """
    def __call__(self, text, positions=True, **kwargs):
        tokens = jieba.cut(text)          # 结巴分词，返回迭代器
        for token in tokens:
            t = Token()                   # 创建 Whoosh 的 Token 对象
            t.text = token                # 设置分词结果
            t.pos = kwargs.get("pos", 0)  # 设置位置信息（可选，默认0）
            yield t                       # 生成器返回 Token

# ============================== 特征提取函数 ==============================
def extract_text(path):
    """
    提取文件的文本特征（描述信息）。
    调用外部模块 describe 的 extract_text_feature 函数。
    """
    text = describe.extract_text_feature(path)   # 假定 describe 已导入
    return text

def extract_img_vector(path):
    """
    提取图像或视频的视觉特征向量。
    若为视频，先提取关键帧；若为图片，直接读取。
    返回所有帧/图的特征向量列表。
    """
    results = []                              # 存储所有特征向量
    # 根据文件扩展名判断类型（视频为 mp4，其他视为图片）
    type_ = "video" if path.endswith("mp4") else "image"
    print(type_)                              # 打印类型（调试用）

    if type_ == "video":
        # 视频：提取关键帧（返回 PIL Image 列表）
        frames = extract_frame.extract_keyframes(path)
    else:
        # 图片：直接打开为单帧列表
        frames = [Image.open(path)]

    for img in frames:
        # 调用 clip_feature 模块提取图像向量（假设返回向量）
        vector = clip_feature.extract_img_vector(img)
        results.append(vector)
    return results

# ============================== 主程序入口（特征入库） ==============================
if __name__ == "__main__":
    # 导入所需的外部模块（放在主程序内，确保依赖已加载）
    import clip_feature          # CLIP 图像特征提取
    import describe              # 文本描述提取
    import qwen3_embedding       # Qwen 文本嵌入
    import extract_audio_text    # 音频文字提取（仅视频）

    # 创建 FAISS 索引（L2 距离）
    img_index = faiss.IndexFlatL2(768)    # 图像特征维度 768
    text_index = faiss.IndexFlatL2(1024)  # 文本特征维度 1024

    # 用于存储特征和路径映射的列表/字典
    text_embeddings = []          # 所有文本向量
    img_embeddings = []           # 所有图像向量
    text_path_dict = {}           # 索引 -> 文件路径（文本）
    img_path_dict = {}            # 索引 -> 文件路径（图像）

    # 创建 Whoosh 索引使用的结巴分词器实例
    jieba_analyzer = JiebaTokenizer()

    # 定义 Whoosh 索引模式：ID 唯一，content 和 audio_text 均存储且分词
    schema = Schema(
        id=ID(unique=True, stored=True),          # 文档 ID（文件路径）
        content=TEXT(stored=True, analyzer=jieba_analyzer),   # 视觉描述文本
        audio_text=TEXT(stored=True, analyzer=jieba_analyzer) # 音频识别文本
    )

    # 创建索引目录（若不存在）
    if not os.path.exists("indexdir"):
        os.mkdir("indexdir")
    # 创建 Whoosh 索引写入器
    # 配置whoosh写入时的相关信息
    ix = create_in("indexdir", schema)
    writer = ix.writer()

    # 待处理的媒体文件根目录（硬编码，请按需修改）
    root_path = "F:\\video\\"
    file_content = []   # 用于保存所有文件的汇总信息（文件名 + 文本 + 音频文本）

    # 遍历目录下所有文件（仅当前层，不递归子目录，因为 os.walk 会递归）
    """
    root：当前正在遍历的文件夹路径
    dirs：当前文件夹里的子文件夹列表
    files：当前文件夹里的所有文件列表
    """
    for root, dirs, files in os.walk(root_path):
        # 读取每一个视频文件的信息,处理单个时评
        for file in tqdm(files, desc="Processing files"):
            file_path = root_path + file   # 完整路径（注意：原代码未拼接子目录，保持原逻辑）

            # ---------- 1. 图像/视频视觉特征（多帧） ----------
            img_vector_list = extract_img_vector(file_path)
            for vector in img_vector_list:
                # 为每个向量分配一个全局索引，并记录对应文件路径
                img_path_dict[len(img_path_dict)] = file_path
                img_embeddings.append(vector)

            # ---------- 2. 文本描述特征 ----------
            # 只是文本描述
            text = extract_text(file_path)   # 调用 describe 模块

            # ---------- 3. 音频文字（仅视频） ----------
            # 提取音频文字内容
            if file_path.endswith("mp4"):
                audio_text = extract_audio_text.extract_audio_text(file_path)
            else:
                audio_text = ""   # 非视频无音频文字

            # 保存文件汇总信息（用于后续参考）
            file_content.append(file + "\t" + text + "\t" + audio_text)

            # 将文本和音频文本写入 Whoosh 索引（ID 为文件路径），构建倒排索引
            writer.add_document(id=file_path, content=text, audio_text=audio_text)

            # ---------- 4. Qwen 文本嵌入向量（基于文本描述） ----------
            text_vector = qwen3_embedding.extract_qwen_embedding(text)
            text_path_dict[len(text_path_dict)] = file_path
            text_embeddings.append(text_vector)

    # ============================== 保存索引和映射文件 ==============================
    # 将图像和文本向量添加到 FAISS 索引
    # 构建向量数据库
    img_index.add(np.array(img_embeddings))
    text_index.add(np.array(text_embeddings))

    # 写入 FAISS 索引文件
    # 保存向量数据库
    faiss.write_index(img_index, "img_index.faiss")
    faiss.write_index(text_index, "text_index.faiss")

    # 提交 Whoosh 索引写入
    # 保存倒排索引数据库
    writer.commit()

    # 报春文件路径
    # 保存文件内容汇总（以制表符分隔）
    with open("file_content", "w", encoding="utf-8") as f:
        f.writelines("\n".join(file_content))
    # 保存路径映射字典（JSON 格式）
    with open("text_path_dict", "w", encoding="utf-8") as f:
        json.dump(text_path_dict, f, ensure_ascii=False)
    with open("img_path_dict", "w", encoding="utf-8") as f:
        json.dump(img_path_dict, f, ensure_ascii=False)

    # （可选）从文件加载索引的示例（被注释）
    # index = faiss.read_index("my_index.faiss")

## 2. 在线处理部分

### 2.1 重排序

In [ ]:
# ============================== 重排序（Reranker）模块 ==============================
# 该模块使用 Qwen3-Reranker 模型对查询和文档进行相关性打分。
# 模型输出 "yes" 或 "no" 的概率，并返回 "yes" 的置信度作为分数。

# ============================== 依赖检查与导入 ==============================
# Requires transformers>=4.51.0
import torch
from modelscope import AutoModel, AutoTokenizer, AutoModelForCausalLM

# ============================== 辅助函数定义 ==============================

def format_instruction(query, doc):
    """
    将查询和文档拼接成模型要求的输入格式。
    """
    output = "<Query>: {query}\n<Document>: {doc}".format(query=query, doc=doc)
    return output


def process_inputs(pairs):
    """
    对输入的文本对进行 tokenization，添加系统提示和特殊 tokens，
    并进行 padding 处理，返回模型所需的张量字典。
    """
    # 先进行基本 tokenization（不填充，不截断，保留最长序列）
    inputs = tokenizer(
        pairs,
        padding=False,                     # 暂不填充
        truncation='longest_first',        # 按最长序列截断
        return_attention_mask=False,       # 后续手动添加
        max_length=max_length - len(prefix_tokens) - len(suffix_tokens)  # 预留系统提示和助手前缀的长度
    )

    # 在每个输入序列前后添加系统提示和助手前缀的 token ID
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = prefix_tokens + ele + suffix_tokens

    # 使用 tokenizer 的 pad 功能进行填充，返回 PyTorch 张量
    inputs = tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=max_length)

    # 将张量移动到模型所在的设备（GPU/CPU）
    for key in inputs:
        inputs[key] = inputs[key].to(model.device)

    return inputs


@torch.no_grad()
def compute_logits(inputs, **kwargs):
    """
    通过模型计算 logits，提取 "yes" 和 "no" 对应的概率，
    返回 "yes" 的置信度（经过 log_softmax 归一化后的指数值）。
    """
    # 模型前向传播，获取最后一个 token 的 logits（形状：[batch, vocab_size]）
    batch_scores = model(**inputs).logits[:, -1, :]

    # 提取 "no" 和 "yes" 对应的 logits
    true_vector = batch_scores[:, token_true_id]   # yes 的 logit
    false_vector = batch_scores[:, token_false_id] # no 的 logit

    # 堆叠成 [batch, 2] 便于计算 softmax
    batch_scores = torch.stack([false_vector, true_vector], dim=1)
    # 计算 log_softmax，然后取 "yes" 的概率（指数化）
    batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
    scores = batch_scores[:, 1].exp().tolist()   # 返回 "yes" 的置信度列表
    return scores


# ============================== 加载模型和分词器 ==============================
# 注意：这里的分词器使用左侧填充（left padding），因为因果语言模型需要左填充以保持位置正确
tokenizer = AutoTokenizer.from_pretrained("F:\\qwen3_reranker", padding_side='left')
# 加载因果语言模型（Qwen3-Reranker），并设置为评估模式
model = AutoModelForCausalLM.from_pretrained("F:\\qwen3_reranker").eval()

# （注释中提供了可选的 flash_attention_2 加速建议，保留原注释）
# We recommend enabling flash_attention_2 for better acceleration and memory saving.
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-Reranker-0.6B", torch_dtype=torch.float16, attn_implementation="flash_attention_2").cuda().eval()

# 获取 "no" 和 "yes" 对应的 token ID（用于后续提取概率）
token_false_id = tokenizer.convert_tokens_to_ids("no")
token_true_id = tokenizer.convert_tokens_to_ids("yes")

# 最大序列长度（Qwen3 支持 8192）
max_length = 8192

# ============================== 构建系统提示和特殊 tokens ==============================
# 系统提示：要求模型只能回答 "yes" 或 "no"
prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
# 后缀：关闭深度思考模式（<think> 标签内为空）
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

# 将前缀和后缀转换为 token ID 列表（不添加特殊 token）
prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)


# ============================== 对外接口函数 ==============================
def reranker(query, document):
    """
    重排序接口：输入查询和文档文本，返回模型给出的相关性分数（0~1 之间）。
    分数越高表示文档与查询越相关。
    """
    # 将查询和文档格式化为模型输入
    queries = [query]
    documents = [document]
    pairs = [format_instruction(query, doc) for query, doc in zip(queries, documents)]

    # Tokenize 并处理输入
    inputs = process_inputs(pairs)

    # 计算分数
    scores = compute_logits(inputs)
    return scores[0]   # 返回单个查询的分数（列表第一个元素）

    # 原注释中有一个 print，已被注释掉，保留原样
    # print("scores: ", scores)

### 2.2 在线处理问答

In [ ]:
# ============================== Flask 应用入口 ==============================
# 该文件实现了一个基于 Flask 的 Web 服务，提供视频检索与问答功能。
# 它集成了多路召回（TF-IDF、CLIP 图像检索、Qwen 文本向量检索）、
# 重排序（Reranker）和视觉语言模型（Qwen2.5-VL）的问答推理。

# ============================== 模块导入 ==============================
from flask import Flask, render_template, request, jsonify   # Flask Web 框架
import os                                                     # 文件和路径操作
from whoosh.qparser import QueryParser                        # Whoosh 查询解析器
from whoosh.index import open_dir                             # 打开 Whoosh 索引
import jieba                                                  # 中文分词（第一次导入）

from extract_feature import *                                 # 导入特征提取模块中的所有函数（包含文本、图像特征等）

import cal_score                                              # 自定义模块：重排序（Reranker）

import faiss                                                  # Facebook 相似性搜索库
import numpy as np                                            # 数值计算
import time                                                   # 时间模块（未实际使用，保留）

import jieba                                                  # 中文分词（重复导入，保留原逻辑）

import json                                                   # JSON 序列化/反序列化
import clip_feature                                           # CLIP 特征提取（图像/文本）
import describe                                               # 描述生成或问答（调用 Qwen2.5-VL）
import qwen3_embedding                                        # Qwen 文本嵌入

# ============================== 加载索引和映射文件 ==============================
# 打开 Whoosh 索引目录（包含分词后的文本索引）
ix = open_dir("indexdir")

# 读取 FAISS 向量索引（文本和图像）
text_index = faiss.read_index("text_index.faiss")   # 文本向量索引（维度 1024）
img_index = faiss.read_index("img_index.faiss")     # 图像向量索引（维度 768）

# 加载路径映射字典（向量索引序号 -> 文件路径）
with open("img_path_dict", "r", encoding="utf-8") as f:   # 原代码缺少编码，补充但保持逻辑不变
    img_path_dict = json.load(f)
with open("text_path_dict", "r", encoding="utf-8") as f:
    text_path_dict = json.load(f)

# ============================== 召回函数定义 ==============================

def tfidf_search(query):
    """
    基于 Whoosh 的 TF-IDF 文本检索（使用结巴分词）。
    返回 [(文件路径, 归一化得分), ...]
    """
    # 对查询进行结巴分词，并用 OR 连接多个词
    words = jieba.cut(query)                     # 分词结果
    parser = QueryParser("content", schema=ix.schema)   # 在 content 字段上解析查询
    query_parsed = parser.parse(text=" OR ".join(words))  # 构建布尔查询

    # 执行搜索
    results = []
    with ix.searcher() as searcher:
        hits = searcher.search(query_parsed)      # 返回匹配文档的迭代器
        for hit in hits:
            results.append([hit["id"], hit.score])  # 记录文档 ID 和原始得分

    # 归一化得分（除以最大得分）
    max_score = max(results, key=lambda s: s[1])[1] if results else 1.0
    results = [[id_, score / max_score] for id_, score in results]
    return results


def convert_d2cos(d):
    """
    将欧氏距离转换为余弦相似度（近似）。
    公式：cos = 1 - d^2 / 2 （适用于单位向量）
    """
    return 1 - d ** 2 / 2


def clip_search(query):
    """
    使用 CLIP 文本编码器将查询转为向量，在图像向量索引中检索。
    返回前 20 个最相似的图像文件及其相似度得分（余弦）。
    """
    vector = clip_feature.extract_text_vector(query)          # 获取查询文本的 CLIP 向量
    distance, ids = img_index.search(np.array([vector]), 20)  # 检索，返回欧氏距离和序号
    # 组合结果：将序号映射到文件路径，并将距离转换为相似度
    result = [[img_path_dict[str(s)], convert_d2cos(d)] for s, d in zip(ids[0], distance[0])]
    return result


def text_search(query):
    """
    使用 Qwen 文本嵌入模型将查询转为向量，在文本向量索引中检索。
    返回前 20 个最相似的文本文件及其相似度得分（余弦）。
    """
    vector = qwen3_embedding.extract_qwen_embedding(query)    # 获取查询的 Qwen 嵌入向量
    distance, ids = text_index.search(np.array([vector]), 20) # 检索
    result = [[text_path_dict[str(s)], convert_d2cos(d)] for s, d in zip(ids[0], distance[0])]
    return result


def build_id_to_docnum_map(ix):
    """
    构建文档 ID（文件路径）到音频文本内容的映射字典。
    用于后续获取召回文档的音频文字信息。
    """
    id_map = {}
    with ix.reader() as reader:
        # 遍历索引中的所有文档
        for docnum in range(reader.doc_count_all()):
            doc = reader.stored_fields(docnum)   # 获取存储的字段
            id_map[doc["id"]] = doc["audio_text"]  # 记录 ID -> audio_text
    return id_map


# 构建 ID 到音频文本的映射（供 qa 函数使用）
id_document_map = build_id_to_docnum_map(ix)


# ============================== 核心问答函数 ==============================
def qa(query):
    """
    多路召回 + 融合 + 重排序 + 视觉问答（Qwen2.5-VL）的完整流程。
    返回一个字符串，包含最终答案和视频文件路径（以换行分隔）。
    """
    # ---------- 1. 三路召回 ----------
    item_list1 = tfidf_search(query)      # TF-IDF 召回
    item_list2 = clip_search(query)       # CLIP 图像召回
    item_list3 = text_search(query)       # Qwen 文本召回

    # ---------- 2. 多路融合（分数累加） ----------
    item_score = {}
    for s, score in item_list1 + item_list2 + item_list3:
        item_score[s] = item_score.get(s, 0) + score

    # 按总分降序排序，取前 3 个候选文档
    item_score = sorted(item_score.items(), key=lambda s: s[1], reverse=True)[0:3]

    # ---------- 3. 对每个候选进行重排序和视觉问答 ----------
    result_score = []
    for item, score in item_score:
        # 获取该文档对应的音频文字（作为背景信息）
        audio_text = id_document_map[item]
        # 构建提示词：包含背景音频文字和用户问题，要求模型判断相关性
        prompt = f"背景信息{audio_text}问题{query} 如果不图片和问题不相关，直接输出不相关，不要输出无关内容"


        """
        对召回的内容
        """
        # 调用 describe.extract_text_feature 进行视觉问答（传入 item 文件路径和 prompt）
        # 注意：这里的 describe 模块可能调用了 Qwen2.5-VL 模型
        result = describe.extract_text_feature(item, prompt)

        # 调用重排序模型（Reranker）对问答结果进行二次打分
        score = cal_score.reranker(query, result)
        result_score.append([result, item, score])

        # 如果得分 >= 0.9，则提前退出循环（认为已找到满意答案）
        if score >= 0.9:
            break

    # ---------- 4. 选择得分最高的作为最终答案 ----------
    # 如果 result_score 为空（理论上不会），则设置默认值
    if not result_score:
        return "未找到相关结果\n"
    answer, item, score = sorted(result_score, key=lambda s: s[-1], reverse=True)[0]

    # 将文件路径转换为静态资源路径（仅保留文件名，放在 static/videos/ 下）
    item = "static\\videos\\" + item.split("\\")[-1]

    # 返回答案和视频路径（用换行分隔）
    return answer + "\n" + item


# ============================== Flask 应用配置 ==============================
app = Flask(__name__)   # 创建 Flask 应用实例

# 确保视频存放目录存在
VIDEO_FOLDER = 'static/videos'
if not os.path.exists(VIDEO_FOLDER):
    os.makedirs(VIDEO_FOLDER)


def process_question(question):
    """
    处理用户问题的包装函数，直接调用 qa 函数。
    """
    result = qa(question)
    return result


# ============================== Flask 路由定义 ==============================

@app.route('/')
def index():
    """首页，渲染 index.html 模板"""
    return render_template('index.html')


@app.route('/api/ask', methods=['POST'])
def ask_question():
    """
    处理用户提问的 API。
    接收 JSON 请求体，包含 'question' 字段，返回答案文本。
    """
    data = request.json
    question = data.get('question', '')
    answer = process_question(question)
    return jsonify({'answer': answer})


@app.route('/api/play-video', methods=['POST'])
def play_video():
    """
    播放视频的 API（安全限制：只能播放 static/videos 目录下的视频）。
    接收 JSON 请求体，包含 'video_path' 字段，返回视频 URL 或错误信息。
    """
    data = request.json
    video_path = data.get('video_path', '')

    # 检查文件是否存在且是支持的视频格式
    if os.path.exists(video_path) and video_path.lower().endswith(('.mp4', '.webm', '.ogg')):
        # 安全检查：确保视频路径在允许的目录下
        if os.path.abspath(video_path).startswith(os.path.abspath(VIDEO_FOLDER)):
            # 计算相对于当前工作目录的相对路径，并转换为 URL 格式
            relative_path = os.path.relpath(video_path, os.getcwd())
            return jsonify({
                'success': True,
                'video_url': '/' + relative_path.replace(os.sep, '/')
            })
        else:
            return jsonify({
                'success': False,
                'error': '安全限制：只能播放指定目录下的视频文件'
            })
    else:
        return jsonify({
            'success': False,
            'error': '视频文件不存在或格式不支持'
        })


# ============================== 启动 Flask 服务 ==============================
if __name__ == '__main__':
    app.run(debug=True)   # 开启调试模式